### Install dependencies

In [11]:
# !pip install torch lightrag-hku transformers sentence-transformers ragas datasets python-dotenv accelerate langchain-community nest_asyncio

### Import libraries 

In [1]:
import os
import json
import glob
import asyncio
import re
from pathlib import Path
from typing import Optional

import torch
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from sentence_transformers import SentenceTransformer

from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from lightrag.llm.openai import openai_complete_if_cache


/opt/anaconda3/envs/HealthCare/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
PAYLOAD_DIR = "payload_for_ref/payloads"
DATASET_ROOT = "medical_datasets/whole_multicare_dataset"
WORKING_DIR = "./lightrag_medgemma_wd"
HF_TOKEN = os.getenv("HF_TOKEN")
MAX_CASES = 100
EVAL_SAMPLES = 10
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MEDGEMMA_MODEL = "google/medgemma-4b-it"

os.makedirs(WORKING_DIR, exist_ok=True)

### Process data

In [14]:
"""Reconstruct image path from .json files"""
def remap_image_path(broken_path: str, dataset_root: str) -> str:
    fname = os.path.basename(broken_path)         
    prefix4 = fname[:4]                            
    prefix5 = fname[:5]                            
    return os.path.join(dataset_root, prefix4, prefix5, fname)

def parse_payload(json_path: str, dataset_root: str) -> dict:
    with open(json_path, "r", encoding="utf_8") as f:
        blocks = json.load(f)

    texts = []
    images = []
    for block in blocks:
        if block.get("type") == "text":
            texts.append(block.get("text", ""))
        elif block.get("type") == "image":
            broken = block.get("img_path", "")
            correct_path = remap_image_path(broken, dataset_root) if broken else ""
            images.append({
                "path": correct_path,
                "caption": " ".join(block.get("image_caption", [])),
                "footnote": " ".join(block.get("image_footnote", []))
            })

    full_text = (os.linesep * 2).join(texts)

    for img in images:
        if img["caption"]:
            full_text += os.linesep + f"[Image] {img['caption']}"
        if img["footnote"]:
            full_text += os.linesep + f"[ImageNote] {img['footnote']}"

    return {"case_id": Path(json_path).stem, "text": full_text, "images": images}

def load_all_payloads(payload_dir: str, dataset_root: str, max_cases: int = MAX_CASES):
    files = sorted(glob.glob(os.path.join(payload_dir, "*.json")))[:max_cases]
    print(f"Loading {len(files)} payload files...")
    cases = [parse_payload(f, dataset_root) for f in files]
    print(f"Loaded {len(cases)} cases.")
    return cases

def load_eval_case(payload_dir: str, dataset_root: str, max_cases: int = MAX_CASES):
    files = sorted(glob.glob(os.path.join(payload_dir, "*.json")))[max_cases:max_cases+max_cases]
    print(f"Loading {len(files)} payload files...")
    eval_cases = [parse_payload(f, dataset_root) for f in files]
    print(f"Loaded {len(cases)} cases.")
    return eval_cases

### MedGemmaWrapper

In [15]:
class MedGemmaWrapper:
    def __init__(self, model_id: str = MEDGEMMA_MODEL, hf_token: Optional[str] = None):
        print(f"Loading MedGemma model.....")
        device = (
            "cuda" if torch.cuda.is_available() else
            "mps" if torch.backends.mps.is_available() else
            "cpu"
        )
        dtype = torch.bfloat16 if device in ("cuda", "mps") else torch.float32

        self.tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, token=hf_token, dtype=dtype, device_map="auto" if device == "cuda" else device, trust_remote_code=True)
        self.device = device

    def generate(self, prompt: str, max_new_tokens: int = 512) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

# Declare medgemma_wrapper object
medgemma_wrapper = MedGemmaWrapper(model_id=MEDGEMMA_MODEL, hf_token=HF_TOKEN)

_llm_lock = asyncio.Lock()
async def medgemma_llm_func(prompt, system_prompt=None, history_messages=[], **kwarg) -> str:
    strict_rules = (
        "\n\nCRITICAL FORMATTING INSTRUCTIONS:\n"
        "- DO NOT wrap entity names, types, or descriptions in angle brackets (< or >).\n"
        "- You must output EXACTLY the requested fields (4 for Entities, 5 for Relations).\n"
        "- Output ONLY the requested format and nothing else."
    )

    prompt = prompt + strict_rules

    full_prompt = ""
    if system_prompt:
        full_prompt += f"<system>" + os.linesep + f"{system_prompt}" + os.linesep + f"</system>" + os.linesep * 2
    for msg in history_messages:
        role = msg.get("role", "user")
        content  = msg.get("content", "")
        full_prompt += f"<{role}>" + os.linesep + f"{content}" + os.linesep + f"</{role}>" + os.linesep * 2
    full_prompt += f"<user>" + os.linesep + f"{prompt}" + os.linesep + f"</user>" + os.linesep + f"<model>" + os.linesep

    loop = asyncio.get_event_loop()
    async with _llm_lock:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(None, medgemma_wrapper.generate, full_prompt)
    if result:
        result = result.replace("<", "").replace(">", "")
    return result



Loading MedGemma model.....


Loading weights: 100%|██████████| 883/883 [00:20<00:00, 43.67it/s] 


### Embedding Function

In [4]:
embed_model = SentenceTransformer(EMBEDDING_MODEL)
_embed_lock = asyncio.Lock()  

async def local_embedding_func(texts: list[str]) -> list[list[float]]:
    async with _embed_lock:
        loop = asyncio.get_event_loop()
        embeddings = await loop.run_in_executor(
            None, lambda: embed_model.encode(texts, normalize_embeddings=True)
        )
    return embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9067.72it/s]


### OpenAIWrapper

In [5]:
async def ollama_openai_wrapper(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        model="llama3.1",
        prompt=prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key="ollama",
        base_url="http://localhost:11434/v1",
        **kwargs
    )


### Build LightRAG

In [6]:
async def build_lightrag(cases: list[dict]) -> LightRAG:
    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=ollama_openai_wrapper,
        embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=512, func=local_embedding_func)
    )

    await rag.initialize_storages()
    
    print(f"Inserting {len(cases)} cases into LightRAG knowledge graph...")
    for i, case in enumerate(cases):
        if case["text"].strip():
            await rag.ainsert(case["text"])
            if (i + 1) % 10 == 0:
                print(f" Inserted {i + 1}/{len(cases)} cases")

    print("Knowledge graph built successfully")
    return rag

async def load_existing_rag() -> LightRAG:
    print(f"Load existing LightRAG from {WORKING_DIR}...")

    rag = LightRAG(
            working_dir=WORKING_DIR,
            llm_model_func=ollama_openai_wrapper,
            embedding_func=EmbeddingFunc(embedding_dim=384, max_token_size=512, func=local_embedding_func)
        )

    await rag.initialize_storages()
    return rag

async def query_rag(rag: LightRAG, question: str, mode: str = "hybrid") -> dict:
    result = await rag.aquery(question, param=QueryParam(mode=mode))
    return {"question": question, "answer": result}



### Evaluation Metric

In [ ]:
async def evaluate_retrieval(cases: list, rag, k: int = 5, n_samples: int = 10):
    eval_data = []
    for case in cases[:n_samples]:
        # 1. Generate the question
        prompt = f"""
        Based ONLY on the following clinical text, write exactly ONE realistic medical question 
        that a doctor might ask, which can be answered by this text. Do not output anything else.
        Text: {case['text'][:1500]}
        """
        realistic_question = await ollama_openai_wrapper(prompt=prompt)
        realistic_question = realistic_question.strip()
        # 2. QUERY THE ACTUAL VECTOR DATABASE
        vdb_results = await rag.chunks_vdb.query(realistic_question, top_k=k)
        
        # 3. FIX: EXTRACT THE RAW TEXT CONTENT instead of the ID
        # (NanoVectorDB usually stores the text inside 'content' or 'text' key)
        retrieved_texts = [res.get('content', res.get('text', '')) for res in vdb_results]
        eval_data.append({
            "ground_truth_text": case["text"],
            "retrieved_texts": retrieved_texts, 
        })
    # 4. CALCULATE THE METRICS 
    recall_at_k_count = 0
    top1_accuracy_count = 0
    mrr_sum = 0.0
    n = len(eval_data)
    for item in eval_data:
        gt_text = item["ground_truth_text"]
        top_k_retrieved_texts = item["retrieved_texts"][:k]
        
        # We check if a significant portion of the ground truth text (e.g. first 50 chars) 
        # is inside the retrieved chunk, or vice versa, to confirm a match.
        match_substring = gt_text[:50].strip()
        # Check Top-1
        if len(top_k_retrieved_texts) > 0 and (match_substring in top_k_retrieved_texts[0] or top_k_retrieved_texts[0][:50] in gt_text):
            top1_accuracy_count += 1
        # Check Recall and MRR
        for rank_idx, retrieved_text in enumerate(top_k_retrieved_texts):
            if match_substring in retrieved_text or retrieved_text[:50] in gt_text:
                recall_at_k_count += 1
                rank = rank_idx + 1
                mrr_sum += 1.0 / rank
                break 
    print(f"Total Queries (n): {n}")
    print(f"Cutoff (k): {k}")
    print(f"Top-1 Accuracy: {top1_accuracy_count / n:.4f}")
    print(f"Recall@{k}: {recall_at_k_count / n:.4f}")
    print(f"MRR@{k}: {mrr_sum / n:.4f}")
    return {"recall": recall_at_k_count / n, "mrr": mrr_sum / n}

In [8]:
# async def evaluate_retrieval(cases: list, k: int = 5, n_samples: int = 10):
#     case_texts = [c["text"] for c in cases]
#     case_ids = [c["case_id"] for c in cases]
#     doc_embeddings = await local_embedding_func(case_texts)

#     eval_data = []

#     for case in cases[:n_samples]:
#         prompt = f"""
#         Based ONLY on the following clinical text, write exactly ONE realistic medical question 
#         that a doctor might ask, which can be answered by this text. Do not output anything else.
#         Text: {case['text'][:1500]}
#         """

#         realistic_question = await ollama_openai_wrapper(prompt=prompt)
#         realistic_question = realistic_question.strip()

#         print(f"Generated Q: {realistic_question[:80]}...")

#         query_embedding = await local_embedding_func([realistic_question])

#         similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

#         # Ranked documents
#         ranked_indices = np.argsort(similarities)[::-1]
#         ranked_case_ids = [case_ids[idx] for idx in ranked_indices]

#         eval_data.append({
#             "ground_truth_id": case["case_id"],
#             "retrieved_ids": ranked_case_ids, 
#         })

#     recall_at_k_count = 0
#     top1_accuracy_count = 0
#     mrr_sum = 0.0
#     n = len(eval_data)

#     for item in eval_data:
#         gt_id = item["ground_truth_id"]
#         top_k_retrieved = item["retrieved_ids"][:k]

#         if len(top_k_retrieved) > 0 and top_k_retrieved[0] == gt_id:
#             top1_accuracy_count += 1

#         if gt_id in top_k_retrieved:
#             recall_at_k_count += 1
#             rank = top_k_retrieved.index(gt_id) + 1
#             mrr_sum += 1.0 / rank

#     print(f"Total Queries (n): {n}")
#     print(f"Cutoff (k): {k}")
#     print(f"Top-1 Accuracy: {top1_accuracy_count / n:.4f} (How often the exact document was #1)")
#     print(f"Recall@{k}: {recall_at_k_count / n:.4f} (How often the document was in the top {k})")
#     print(f"MRR@{k}: {mrr_sum / n:.4f} (Average rank quality)")

#     return {"recall": recall_at_k_count / n, "mrr": mrr_sum / n}

async def evaluate_rag(cases: list, rag, n_samples: int = 5):
    total_score = 0
    valid_evals = 0

    for case in cases[:n_samples]:
        prompt_q = f"Based ONLY on this text, write one realistic medical question: {case['text'][:1000]}"
        question = await ollama_openai_wrapper(prompt=prompt_q)
        question = question.strip()

        print(f"\nQ: {question}...")

        # Evaluate LightRAG
        rag_response = await query_rag(rag, question, mode="hybrid")
        lightrag_answer = rag_response['answer'] if rag_response and rag_response.get('answer') else "No answer."

        # Use LLM to evaluate answers
        eval_prompt = f"""
        You are a medical grader.
        GROUND TRUTH TEXT: {case['text'][:1500]}
        QUESTION: {question}
        LIGHTRAG'S ANSWER: {lightrag_answer}
        
        Grade LightRAG's answer from 1 to 10 on Accuracy. 
        Did LightRAG successfully use the graph to find the right answer?
        Output ONLY a single integer from 1 to 10.
        """

        try:
            score_response = await ollama_openai_wrapper(prompt=eval_prompt)
            match = re.search(r'\b(10|[1-9])\b', score_response)
            if match:
                score = int(match.group(0))
                total_score += score
                valid_evals += 1
                print(f"-> LightRAG Score: {score}/10")
            else:
                print(f"-> Failed to parse score")
        except Exception as e:
            print(f"-> Error: {e}")

    avg_score = (total_score / valid_evals) if valid_evals > 0 else 0
    print("\n===== FINAL LIGHTRAG GRAPH METRICS =====")
    print(f"LightRAG Hybrid Accuracy: {avg_score:.2f} / 10.0")
    
    return avg_score



### To build Light RAG

In [18]:
cases = load_all_payloads(PAYLOAD_DIR, DATASET_ROOT, max_cases=MAX_CASES)
# rag = await build_lightrag(cases)

Loading 100 payload files...
Loaded 100 cases.


### Print evaluation metrics

In [20]:
eval_cases = load_eval_case(PAYLOAD_DIR, DATASET_ROOT, max_cases=MAX_CASES)
rag_instance = await load_existing_rag()
eval_results = await evaluate_retrieval(cases, rag_instance)
# avg_score = await evaluate_rag(cases, rag_instance)

INFO: [] Loaded graph from ./lightrag_medgemma_wd/graph_chunk_entity_relation.graphml with 3404 nodes, 2180 edges


Loading 100 payload files...
Loaded 100 cases.
Load existing LightRAG from ./lightrag_medgemma_wd...


INFO:nano-vectordb:Load (3404, 384) data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_entities.json'} 3404 data
INFO:nano-vectordb:Load (2180, 384) data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_relationships.json'} 2180 data
INFO:nano-vectordb:Load (295, 384) data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_chunks.json'} 295 data
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.52it/s]

Total Queries (n): 10
Cutoff (k): 5
Top-1 Accuracy: 0.7000
Recall@5: 0.9000
MRR@5: 0.7667


In [ ]:
# async def main():
#     cases = load_all_payloads(PAYLOAD_DIR, DATASET_ROOT, max_cases=MAX_CASES)

#     rag = await build_lightrag(cases)

#     test_questions = [
#         "What are the common symptoms of brain tumor in the reported cases?",
#         "What imaging findings are described in the clinical cases?",
#         "What treatment approaches are mentioned in the case reports?",
#     ]

#     for q in test_questions:
#         result = await query_rag(rag, q, mode="hybrid")

#         if result and result.get('answer'):
#             answer_text = str(result['answer'])
#         else:
#             answer_text = "ERROR: LightRAG returned None."
            
#         print(f"\nQ: {q}\nA: {answer_text[:500]}...\n{'-'*60}")

#     results_df = await evaluate_retrieval_performance(cases, k=5, n_samples=EVAL_SAMPLES)

#     return rag, results_df

# if __name__ == "__main__":
#     rag_instance, eval_results = await main()

INFO: [] Created new empty graph file: ./lightrag_medgemma_wd/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './lightrag_medgemma_wd/vdb_chunks.json'} 0 data
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 4899 KV load full_docs with 0 records
INFO: [] Process 4899 KV load text_chunks with 0 records
INFO: [] Process 4899 KV load full_entities with 0 records
INFO: [] Process 4899 KV loa

Loading 100 payload files...
Loaded 100 cases.
Inserting 100 cases into LightRAG knowledge graph...


INFO: Processing 1 document(s)
INFO: Parsing (native): doc-be67c6a452b3283e6cc3d1181db49725
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-be67c6a452b3283e6cc3d1181db49725
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: doc-be67c6a452b3283e6cc3d1181db49725
INFO: extract LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: default:extract:f5cfb5e5d620eec0524b10c0cc19c436
INFO:  == LLM cache == saving: default:extract:4728b1b8a838fe1f5b34fe851c661944
INFO: Chunk 1 of 1 extracted 11 Ent + 0 Rel doc-be67c6a452b3283e6cc3d1181db49725-chunk-000
INFO: Merging stage 1/1: unknown_source
INFO: Phase 0: Writing recovery indexes for doc-be67c6a452b3283e6cc3d1181db49725: 11 candidate entities, 0 candidate relations
INFO: Phase 1: Processing 11 entities from doc-be67c6a452b3283e6cc3d1181db49725 (async: 8)
INFO: Phase 2: Processing 0 relations from doc-be67c6a452b3283e6cc3d1181db49

 Inserted 10/100 cases


INFO:  == LLM cache == saving: default:extract:3c4557115cb69bae7275608049d75290
INFO:  == LLM cache == saving: default:extract:d3f1440bf682b2fe357eadbe1f2764c2
INFO:  == LLM cache == saving: default:extract:83ee263ea5db2c5f2f6d23b9e60bd10f
INFO:  == LLM cache == saving: default:extract:25a1021cb2f99783cedf4decfc97e35a
INFO:  == LLM cache == saving: default:extract:780827d460e91f32efed5beba987d3b0
INFO: Chunk 1 of 4 extracted 8 Ent + 8 Rel doc-cd24aad1fdc5de922a50ab7953f43a3f-chunk-003
INFO:  == LLM cache == saving: default:extract:98bf9af3a9512bbf919e0d496624ba38
INFO: Chunk 2 of 4 extracted 15 Ent + 31 Rel doc-cd24aad1fdc5de922a50ab7953f43a3f-chunk-002
INFO:  == LLM cache == saving: default:extract:63cc5bd890df10d00e498a1032b45b5d
INFO: Chunk 3 of 4 extracted 21 Ent + 19 Rel doc-cd24aad1fdc5de922a50ab7953f43a3f-chunk-001
INFO:  == LLM cache == saving: default:extract:5fd763dc6a7e77b4e059a1e1a11be749
INFO: Chunk 4 of 4 extracted 26 Ent + 0 Rel doc-cd24aad1fdc5de922a50ab7953f43a3f-chunk

...


Q: What imaging findings are described in the clinical cases?
A: Based on the provided text, the following imaging findings are described:

1. Thoracic computed tomography (CT) at admission showing an aortic filling defect.
2. Thoracic CT on the second day of hospitalization showing changes in the aortic filling defect.
3. Thoracic CT at discharge showing no aggravation of the aortic filling defect.
4. Bilateral patchy peripheral opacities on chest x-ray, left greater than right.
5. Computed tomography (CT) findings of lymphadenopathy over cervical, bilateral...
------------------------------------------------------------


INFO:  == LLM cache == saving: hybrid:keywords:54fb3bf219fafb0308698f1ec6d128f7
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.21it/s]
INFO: Query nodes: case reports (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 30 relations
INFO: Query edges: treatment approaches, case reports (top_k:40, cosine:0.2)
INFO: Global query: 61 entites, 40 relations
INFO: Raw search results: 97 entities, 66 relations, 0 vector chunks
INFO: After truncation: 97 entities, 66 relations
INFO: Selecting 195 from 195 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 66 relations
INFO: Round-robin merged chunks: 195 -> 195 (deduplicated 0)
INFO: Final context: 97 entities, 66 relations, 20 chunks
INFO: Final chunks S+F/O: E1/1 E2/2 E1/3 E1/4 E2/5 E2/6 E5/7 E2/8 E1/9 E1/10 E2/11 E1/12 E2/13 E3/14 E2/15 E1/16 E2/17 E2/18 E1/19 E2/20
INFO:  == LLM cache == saving: hybrid:query:608634f09c89ebd09e04bcf637cbb76a



Q: What treatment approaches are mentioned in the case reports?
A: Based on the provided case reports, the following treatment approaches are mentioned:

1. **Rituximab infusion**: In one case report, a 500 mg rituximab infusion was administered to a patient with pemphigus vulgaris.
2. **Liposomal amphotericin B**: In another case report, a patient diagnosed with sino-orbital mucormycosis was treated with liposomal amphotericin B 300 mg.
3. **Antiviral therapy with ACE-inhibitors and steroids**: Although not explicitly mentioned, the use of antiviral therapy wi...
------------------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 170.06it/s]

n: 10
k: 5
recall@k: 0.0
top1_accuracy: 0.0
mrr@k: 0.0
